# Detecção de Defeitos em PCBs utilizando RT-DETR

>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira

## 1. Introdução e Motivação

> **Resumo do Estudo:** Este trabalho apresenta o desenvolvimento de um sistema de Visão Computacional para detecção automática de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura **RT-DETR** (Real-Time DEtection TRansformer). Para viabilizar a execução local e manter comparação consistente entre modelos, o conjunto de dados foi organizado preservando a distribuição das classes de defeito. Os resultados obtidos são comparados com os modelos YOLOv11, Faster R-CNN e RetinaNet.

### 1.1 Contextualização
No cenário da Indústria 4.0, a garantia de qualidade na fabricação de componentes eletrônicos é crítica para reduzir perdas, retrabalho e falhas em campo. As Placas de Circuito Impresso (PCBs) são a base de praticamente todos os dispositivos eletrônicos modernos. Com a miniaturização dos componentes, a inspeção visual tornou-se mais complexa e exige soluções automáticas robustas.

### 1.2 O Problema
Tradicionalmente, a inspeção de PCBs é realizada de forma manual por operadores humanos ou por algoritmos de visão clássica baseados em regras rígidas. Esses métodos apresentam limitações no ambiente industrial:
>* **Fadiga Humana:** A inspeção visual repetitiva aumenta a chance de erro e inconsistências.
>* **Baixa Escalabilidade:** A inspeção manual é lenta e cria gargalos na linha de produção.
>* **Sensibilidade de Regras:** Métodos clássicos falham com variações de iluminação, rotação e ruído.

### 1.3 A Solução Proposta
Para reduzir esses problemas e automatizar o processo de inspeção, este notebook adota Deep Learning com **RT-DETR** (Real-Time DEtection TRansformer), uma arquitetura baseada em Transformers que combina a precisão dos detectores baseados em atenção com a velocidade necessária para aplicações em tempo real.

O objetivo é identificar e localizar seis tipos comuns de defeitos de fabricação:
>1.  **Missing Hole** (Furo faltante)
>2.  **Mouse Bite** (Mordida de rato/Falha na borda)
>3.  **Open Circuit** (Circuito aberto)
>4.  **Short** (Curto-circuito)
>5.  **Spur** (Esporão/Rebarba)
>6.  **Spurious Copper** (Cobre residual)

A aplicação proposta busca aumentar a eficiência do controle de qualidade industrial, reduzindo desperdícios de material e o risco de envio de placas defeituosas.

## 2. Análise Exploratória dos Dados (EDA)
> Notebook pode ser encontrado em ./EDA_VC.ipynb

> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

In [1]:
import importlib.util
import subprocess
import sys

FORCE_REINSTALL_TORCH = False
PREFER_CUDA_ON_NVIDIA = True
CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu121"

def _run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args)

def _module_exists(module_name):
    return importlib.util.find_spec(module_name) is not None

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

def _torch_stack_ready(needs_cuda):
    required_modules = ["torch", "torchvision", "torchaudio"]
    if not all(_module_exists(module_name) for module_name in required_modules):
        return False

    import torch

    if needs_cuda:
        return torch.cuda.is_available() and (torch.version.cuda is not None)
    return True

def _install_torch_stack(needs_cuda):
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "torch",
        "torchvision",
        "torchaudio",
    ]
    if needs_cuda:
        install_cmd += ["--index-url", CUDA_INDEX_URL]

    try:
        _run_cmd(install_cmd)
    except subprocess.CalledProcessError:
        print("Primeira tentativa falhou. Limpando stack PyTorch e tentando novamente...")
        _run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
        _run_cmd(install_cmd)

nvidia_gpu_detected = _has_nvidia_gpu()
needs_cuda = PREFER_CUDA_ON_NVIDIA and nvidia_gpu_detected

# Garante ferramentas básicas de build/instalação no venv recém-criado.
_run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

if FORCE_REINSTALL_TORCH or not _torch_stack_ready(needs_cuda):
    target_label = "CUDA 12.1 (cu121)" if needs_cuda else "CPU"
    print(f"Instalando stack PyTorch para {target_label}...")
    _install_torch_stack(needs_cuda)
    print("Stack PyTorch instalada/atualizada.")
else:
    print("Stack PyTorch já compatível com este ambiente.")

required_packages = {
    "pycocotools": "pycocotools",
    "pyyaml": "yaml",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "ultralytics": "ultralytics",
    "Pillow": "PIL",
    "numpy": "numpy",
    "certifi": "certifi",
}

missing = [pkg for pkg, module in required_packages.items() if not _module_exists(module)]
if missing:
    _run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", *missing])
    print("Dependências instaladas:", ", ".join(missing))
else:
    print("Dependências já instaladas.")

import torch

print(f"Torch: {torch.__version__} | CUDA build: {torch.version.cuda} | cuda_available={torch.cuda.is_available()}")
if needs_cuda and not torch.cuda.is_available():
    print("ATENÇÃO: GPU NVIDIA detectada, mas CUDA indisponível. Reinicie o kernel e execute novamente esta célula.")

$ /home/alexandre-oliveira/miniconda3/envs/pcb-defect/bin/python -m pip install --upgrade pip setuptools wheel
Stack PyTorch já compatível com este ambiente.
Dependências já instaladas.
Torch: 2.5.1 | CUDA build: 12.1 | cuda_available=True


In [2]:
# Diagnóstico rápido do runtime PyTorch
import subprocess
import torch

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

nvidia_gpu_detected = _has_nvidia_gpu()

print(f"GPU NVIDIA detectada: {nvidia_gpu_detected}")
print(f"Torch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"cuda_available: {torch.cuda.is_available()}")

if nvidia_gpu_detected and not torch.cuda.is_available():
    print("ATENÇÃO: há GPU NVIDIA, porém CUDA não está ativa. Reexecute a célula anterior e reinicie o kernel.")
elif (not nvidia_gpu_detected) and torch.cuda.is_available():
    print("Observação: CUDA ativa, mas nvidia-smi não foi detectado no PATH.")
else:
    print("Ambiente de execução coerente para seguir com o notebook.")

GPU NVIDIA detectada: True
Torch: 2.5.1
CUDA build: 12.1
cuda_available: True
Ambiente de execução coerente para seguir com o notebook.


In [3]:
import os
import json
from datetime import datetime
from pathlib import Path

import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import RTDETR
from PIL import Image
import numpy as np
import torchvision
from common_detection_protocol import evaluate_ultralytics_coco

# Validação rápida do operador NMS (detecta incompatibilidade torch/torchvision)
try:
    _ = torchvision.ops.nms(
        torch.tensor([[0.0, 0.0, 1.0, 1.0]]),
        torch.tensor([0.9]),
        0.5,
    )
    print("Operador torchvision::nms OK.")
except Exception as exc:
    raise RuntimeError(
        "Falha no operador torchvision."
    ) from exc

Operador torchvision::nms OK.


In [4]:
# Configuração de caminhos
PROJECT_ROOT = Path.cwd()
BASE_DIR = PROJECT_ROOT / "pcb-defect-subset-5000"
RUNS_ROOT = PROJECT_ROOT / "runs" / "detect"

PROJECT_RUN_DIR = RUNS_ROOT / "tcc_pcb_defect_detection" / "rtdetr"
PROJECT_RUN_DIR.mkdir(parents=True, exist_ok=True)

train_images_dir = BASE_DIR / "train" / "images"
val_images_dir = BASE_DIR / "val" / "images"
test_images_dir = BASE_DIR / "test" / "images"

if not train_images_dir.exists():
    raise FileNotFoundError(f"Pasta de treino não encontrada: {train_images_dir}")

data_yaml = {
    "path": str(BASE_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "mouse_bite",
        1: "spur",
        2: "missing_hole",
        3: "short",
        4: "open_circuit",
        5: "spurious_copper",
    },
}

yaml_path = BASE_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Dataset base: {BASE_DIR}")
print(f"Arquivo YAML: {yaml_path}")
print(f"Diretório de saída: {PROJECT_RUN_DIR}")

Dataset base: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000
Arquivo YAML: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/data.yaml
Diretório de saída: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/rtdetr


In [5]:
# ==============================================================================
# CHECAR DATA LEAKS
# ==============================================================================

def check_filename_leakage(train_dir, val_dir):
    # Pega apenas os nomes dos arquivos
    train_files = set(os.listdir(train_dir))
    val_files = set(os.listdir(val_dir))

    print(f"Total Treino: {len(train_files)}")
    print(f"Total Validação: {len(val_files)}")

    # Checa duplicatas exatas de nome
    duplicates = train_files.intersection(val_files)
    if duplicates:
        print(f"PERIGO: {len(duplicates)} arquivos têm EXATAMENTE o mesmo nome em Treino e Validação!")
        print(list(duplicates)[:5])
    else:
        print("Nomes de arquivos exatos não se repetem.")

    #  Checa vazamento por Prefixo (Assumindo que o prefixo indica a placa de origem)
    train_prefixes = set([f.split('_')[0] for f in train_files])
    val_prefixes = set([f.split('_')[0] for f in val_files])

    prefix_leak = train_prefixes.intersection(val_prefixes)

    if prefix_leak:
        print(f"ATENÇÃO: {len(prefix_leak)} placas originais (prefixos) aparecem em AMBOS os conjuntos.")
        print(f"Exemplos: {list(prefix_leak)[:5]}")
    else:
        print("Prefixos distintos. Parece que as placas foram separadas corretamente.")


if os.path.exists(train_images_dir) and os.path.exists(val_images_dir):
    check_filename_leakage(train_images_dir, val_images_dir)

Total Treino: 3973
Total Validação: 532
Nomes de arquivos exatos não se repetem.
ATENÇÃO: 3 placas originais (prefixos) aparecem em AMBOS os conjuntos.
Exemplos: ['light', 'l', 'rotation']


## 3. Arquitetura do Modelo: RT-DETR

Neste estudo, o modelo **RT-DETR** (Real-Time DEtection TRansformer) é adotado como detector baseado em Transformers para inspeção automática de PCBs, combinando alta precisão com capacidade de inferência em tempo real.

Diferente dos detectores tradicionais baseados em convoluções (como YOLO ou Faster R-CNN), o RT-DETR utiliza mecanismos de **atenção** (self-attention) para capturar relações globais na imagem, eliminando a necessidade de componentes como NMS (Non-Maximum Suppression) no pós-processamento.

### Por que RT-DETR para PCBs?
A escolha desta arquitetura considera três pontos relevantes para controle de qualidade industrial:

>* **Atenção Global:** O mecanismo de Transformer captura dependências de longo alcance na imagem, ideal para detectar defeitos que dependem do contexto global da placa.
>* **End-to-End:** Eliminação do NMS resulta em pipeline mais simples e previsível, importante para ambientes industriais.
>* **Velocidade em Tempo Real:** Apesar de usar Transformers, o RT-DETR foi otimizado para manter velocidade competitiva com detectores CNN tradicionais.

### Estrutura Simplificada
O fluxo do RT-DETR pode ser resumido nas etapas abaixo:

>1. **Input:** Imagem da PCB é processada para extração de características.
>2. **Backbone (ResNet/HGNetv2):** Extrai mapas de características multiescala.
>3. **Hybrid Encoder:** Combina características intra-escala (AIFI) e cross-escala (CCFM) usando atenção.
>4. **Transformer Decoder:** Processa object queries para predizer diretamente as detecções.
>5. **Saídas finais:** Classe do defeito e *bounding box* refinada, sem necessidade de NMS.

<div align="center">
  <h3>Arquitetura RT-DETR</h3>
  <img src="https://cdn.jsdelivr.net/gh/ultralytics/assets@main/docs/baidu-rtdetr-model-overview.avif" width="760" alt="Diagrama RT-DETR">
</div>

### O Diferencial do RT-DETR: Hybrid Encoder
O RT-DETR introduz um codificador híbrido eficiente que processa características multiescala em dois estágios:

1. **AIFI (Attention-based Intra-scale Feature Interaction):** Aplica self-attention dentro de cada escala para capturar relações espaciais.
2. **CCFM (CNN-based Cross-scale Feature-fusion Module):** Funde informações entre diferentes escalas usando convoluções, mantendo eficiência computacional.

Essa formulação permite ao RT-DETR alcançar precisão superior aos detectores YOLO em muitos benchmarks, mantendo velocidade comparável.

In [6]:
# Treinamento do modelo RT-DETR-L
if torch.cuda.is_available():
    device_id = 0
    workers = 4
    batch_size = 4
    print(f"Executando em CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device_id = "mps"
    workers = 4
    batch_size = 4
    print("Executando em Apple Silicon MPS")
else:
    device_id = "cpu"
    workers = 2
    batch_size = 2
    print("Executando em CPU")

INPUT_SIZE = 640
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_tag = "standardized_protocol"
run_name = (
    f"{run_timestamp}_rtdetr-l_img{INPUT_SIZE}_e{MAX_EPOCHS}_"
    f"bs{batch_size}_seed42_{run_tag}"
)

model = RTDETR("rtdetr-l.pt")
print("Iniciando treinamento RT-DETR-L")
print(f"Nome do experimento: {run_name}")

results = model.train(
    data=str(yaml_path),
    epochs=MAX_EPOCHS,
    patience=EARLY_STOP_PATIENCE,
    imgsz=INPUT_SIZE,
    batch=batch_size,
    project=str(PROJECT_RUN_DIR),
    name=run_name,
    workers=workers,
    lr0=0.0001,
    device=device_id,
    augment=True,
    verbose=True,

    # Otimização específica da arquitetura
    optimizer="AdamW",
    weight_decay=0.0001,

    # Augmentation moderado compartilhado com os detectores TorchVision
    hsv_h=0.01,
    hsv_s=0.20,
    hsv_v=0.20,
    degrees=10.0,
    translate=0.05,
    scale=0.10,
    fliplr=0.50,
    flipud=0.50,
    perspective=0.0,
    mosaic=0.0,
    mixup=0.0,
    erasing=0.0,

    seed=42,
    deterministic=True,
    amp=False,
    cache=False,
    pretrained=True,
    val=True,
)

results_dir = Path(results.save_dir) if hasattr(results, "save_dir") else Path(str(results))
print(f"Treinamento concluído. Resultados em: {results_dir}")

Executando em CUDA: NVIDIA GeForce GTX 1060 6GB
Iniciando treinamento RT-DETR-L
Nome do experimento: 20260830_2138_rtdetr-l_img640_e100_bs4_seed42_standardized_protocol
New https://pypi.org/project/ultralytics/8.4.135 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.48 🚀 Python-3.11.16 torch-2.5.1 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6063MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, 

/home/alexandre-oliveira/miniconda3/envs/pcb-defect/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      5.13G      1.803     0.6433     0.5005          2        640: 100% ━━━━━━━━━━━━ 993/993 1.1it/s 15:46<1.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.6s0.5ss
                   all        531        751    2.3e-05     0.0262    1.1e-05   1.86e-06

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/home/alexandre-oliveira/miniconda3/envs/pcb-defect/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      5.51G      1.529     0.5314     0.3695          6        640: 100% ━━━━━━━━━━━━ 993/993 1.1it/s 15:36<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.4s0.5ss
                   all        531        751    0.00128     0.0495   0.000128   1.53e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      1.96G      1.658     0.4003     0.3743         11        640: 0% ──────────── 0/993  1.1s

...

     91/100      5.42G     0.3162     0.3704     0.0487          6        640: 100% ━━━━━━━━━━━━ 993/993 1.1it/s 15:34<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.5s0.5ss
                   all        531        751      0.979      0.981      0.987      0.592

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     92/100      2.04G     0.2271     0.3138    0.03144          8        640: 0% ──────────── 0/993  1.0s

/home/alexandre-oliveira/miniconda3/envs/pcb-defect/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     92/100      5.43G      0.317     0.3691    0.04888          8        640: 100% ━━━━━━━━━━━━ 993/993 1.1it/s 15:34<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.5s0.5ss
                   all        531        751      0.979      0.986      0.985      0.591

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     93/100      2.02G     0.2243     0.3226    0.03008          8        640: 0% ──────────── 0/993  1.0s

/home/alexandre-oliveira/miniconda3/envs/pcb-defect/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     93/100      5.43G     0.3176       0.37    0.04908          4        640: 100% ━━━━━━━━━━━━ 993/993 1.1it/s 15:35<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.5s0.5ss
                   all        531        751      0.983      0.982      0.987      0.594
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 83, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

93 epochs completed in 24.956 hours.
Optimizer stripped from /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/rtdetr/20260830_2138_rtdetr-l_img640_e100_bs4_seed42_standardized_protocol/weights/last.pt, 66.2MB
Optimizer stripped from /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-lin

## 4. Análise de Métricas de Desempenho

Para validar a eficácia do RT-DETR na detecção de defeitos em PCBs, utilizamos um conjunto de métricas alinhado ao padrão adotado nos notebooks do YOLO, Faster R-CNN e RetinaNet, com foco em interpretação prática para inspeção industrial.

### Matriz de Confusão
A matriz de confusão mostra, classe por classe, quais defeitos foram corretamente identificados e onde ocorreram confusões.

- **Impacto industrial:** ajuda a identificar erros críticos, como falsos negativos em defeitos que não podem escapar da inspeção.

### Precisão (Precision)
A precisão responde: **"de todos os defeitos que o modelo apontou, quantos eram reais?"**

$$\text{Precision} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Positivos}}$$

### Recall (Sensibilidade)
O recall responde: **"de todos os defeitos existentes, quantos o modelo encontrou?"**

$$\text{Recall} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Negativos}}$$

### F1-Score
O F1-Score equilibra precisão e recall em um único indicador.

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### mAP (Mean Average Precision)
- **mAP@50:** indica a qualidade geral de detecção com critério IoU mais permissivo.
- **mAP@50-95:** métrica mais rigorosa, útil para avaliar a qualidade fina de localização das caixas.

In [7]:
print("Executando avaliação COCO comum no conjunto de teste")
run_dir = Path(results.save_dir)
best_checkpoint = run_dir / "weights" / "best.pt"
test_annotations_path = test_images_dir / "test_annotations.json"

if not best_checkpoint.exists():
    raise FileNotFoundError(f"Melhor checkpoint não encontrado: {best_checkpoint}")
if not test_annotations_path.exists():
    raise FileNotFoundError(f"Anotação COCO de teste não encontrada: {test_annotations_path}")

best_model = RTDETR(str(best_checkpoint))
class_names = [data_yaml["names"][index] for index in sorted(data_yaml["names"])]
metrics_summary = evaluate_ultralytics_coco(
    model=best_model,
    images_dir=test_images_dir,
    annotation_path=test_annotations_path,
    class_names=class_names,
    device=device_id,
    input_size=INPUT_SIZE,
    output_json_path=run_dir / "coco_test_predictions.json",
)
metrics_summary.update(
    {
        "run_dir": str(run_dir),
        "checkpoint": str(best_checkpoint),
        "evaluation_backend": "pycocotools.COCOeval",
    }
)

with open(run_dir / "metrics_summary.json", "w", encoding="utf-8") as file:
    json.dump(metrics_summary, file, indent=2)

print("\n--- Métricas comuns de teste ---")
print(f"mAP@50: {metrics_summary['map50']:.4f}")
print(f"mAP@50-95: {metrics_summary['map50_95']:.4f}")
print(f"Precision macro: {metrics_summary['precision_macro']:.4f}")
print(f"Recall macro: {metrics_summary['recall_macro']:.4f}")
print(f"F1 macro: {metrics_summary['f1_macro']:.4f}")
display(pd.DataFrame(metrics_summary["per_class"]))

csv_path = run_dir / "results.csv"
if csv_path.exists():
    history_df = pd.read_csv(csv_path)
    history_df.columns = history_df.columns.str.strip()
    map50_column = "metrics/mAP50(B)" if "metrics/mAP50(B)" in history_df else "metrics/mAP50"
    map95_column = "metrics/mAP50-95(B)" if "metrics/mAP50-95(B)" in history_df else "metrics/mAP50-95"

    plt.figure(figsize=(9, 5))
    plt.plot(history_df["epoch"], history_df[map50_column], label="mAP@50")
    plt.plot(history_df["epoch"], history_df[map95_column], label="mAP@50-95")
    plt.xlabel("Época")
    plt.ylabel("Métrica de validação")
    plt.title("Evolução das métricas durante o treinamento")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

Executando avaliação COCO comum no conjunto de teste
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.39s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.87s).
Accumulating evaluation results...
DONE (t=0.51s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.525
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.984
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.466
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.514
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.560
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.295
 Average

,category_id,class_name,precision,recall,f1,true_positives,false_positives,false_negatives,support
0,1,mouse_bite,0.933884,0.991228,0.961702,113,8,1,114
1,2,spur,0.810127,0.984615,0.888889,128,30,2,130
2,3,missing_hole,0.888158,0.992647,0.937500,135,17,1,136
3,4,short,0.907801,0.984615,0.944649,128,13,2,130
4,5,open_circuit,0.953947,0.979730,0.966667,145,7,3,148
5,6,spurious_copper,0.873494,0.993151,0.929487,145,21,1,146


<Figure size 900x500 with 1 Axes>

## 5. Testes de Inferência

A etapa de inferência foi separada para o notebook `RT-DETR_inferencia.ipynb`, mantendo este arquivo focado em preparação, treinamento e análise de métricas.

Após finalizar um novo treinamento, utilize o melhor checkpoint (`best.pt`) da execução mais recente para validar o comportamento em imagens de teste e em imagens externas.

## 6. Conclusão

O notebook está configurado para retreinar o **RT-DETR-L** com o protocolo experimental padronizado: entrada de 640 × 640 pixels, augmentation moderado, limite máximo de 100 épocas, early stopping e seleção do melhor checkpoint pela validação.

A avaliação final utiliza o mesmo backend `pycocotools.COCOeval` adotado para as demais arquiteturas. Os resultados numéricos anteriores foram removidos desta conclusão e devem ser preenchidos somente após a execução integral do novo treinamento.

# 7. Referências  

- Zhao, Y., Lv, W., Xu, S., et al. (2024). DETRs Beat YOLOs on Real-time Object Detection. CVPR 2024.
- Ultralytics RT-DETR Documentation: https://docs.ultralytics.com/models/rtdetr/
- PCB Defect Dataset: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset